In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

df = pd.read_parquet(PROCESSED_DIR / "matches_clean.parquet")
df = df.sort_values("date").reset_index(drop=True)

print(f"Loaded {len(df)} matches")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

C:\Users\vasan\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loaded 32287 matches
Date range: 1990-01-12 to 2026-06-10


In [2]:
def calculate_elo(df,
                  initial_rating=1500,
                  k_friendly=20,
                  k_competitive=40,
                  k_worldcup=60):
    """
    Calculate Elo ratings for every match.
    Records ratings BEFORE each match so we never leak future data.
    Higher K = bigger rating change per match.
    WC matches get highest K — they matter most.
    """
    ratings = {}
    history = []

    def get_rating(team):
        return ratings.get(team, initial_rating)

    def expected(rating_a, rating_b):
        return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

    def get_k(row):
        if row["is_worldcup"]:
            return k_worldcup
        elif row["is_friendly"]:
            return k_friendly
        else:
            return k_competitive

    for _, row in df.iterrows():
        home = row["home_team"]
        away = row["away_team"]

        # ratings BEFORE this match
        home_elo = get_rating(home)
        away_elo = get_rating(away)

        # expected outcome
        home_exp = expected(home_elo, away_elo)
        away_exp = 1 - home_exp

        # actual outcome
        if row["outcome"] == "home_win":
            home_act, away_act = 1.0, 0.0
        elif row["outcome"] == "away_win":
            home_act, away_act = 0.0, 1.0
        else:
            home_act, away_act = 0.5, 0.5

        k = get_k(row)

        # store pre-match state
        history.append({
            "date":            row["date"],
            "home_team":       home,
            "away_team":       away,
            "home_elo_before": round(home_elo, 1),
            "away_elo_before": round(away_elo, 1),
            "elo_diff":        round(home_elo - away_elo, 1),
            "home_exp_prob":   round(home_exp, 4),
            "tournament":      row["tournament"],
            "is_worldcup":     row["is_worldcup"],
            "outcome":         row["outcome"],
            "home_score":      row["home_score"],
            "away_score":      row["away_score"],
        })

        # update ratings AFTER match
        ratings[home] = home_elo + k * (home_act - home_exp)
        ratings[away] = away_elo + k * (away_act - away_exp)

    return pd.DataFrame(history), ratings


elo_df, final_ratings = calculate_elo(df)
print(f"Elo calculated for {len(elo_df)} matches")
print(f"Teams rated: {len(final_ratings)}")

Elo calculated for 32287 matches
Teams rated: 326


In [3]:
ratings_df = pd.DataFrame([
    {"team": team, "elo": round(rating, 1)}
    for team, rating in final_ratings.items()
]).sort_values("elo", ascending=False).reset_index(drop=True)

ratings_df.index += 1  # start rank from 1
print("Top 30 teams by Elo (as of June 10, 2026):")
print(ratings_df.head(30).to_string())

Top 30 teams by Elo (as of June 10, 2026):
             team     elo
1           Spain  2054.0
2       Argentina  2035.7
3          France  1989.2
4         England  1950.1
5        Portugal  1934.9
6           Japan  1926.3
7         Morocco  1926.0
8         Germany  1921.6
9          Brazil  1920.2
10       Colombia  1915.0
11    Netherlands  1905.9
12        Ecuador  1901.7
13         Mexico  1888.3
14        Croatia  1874.1
15         Turkey  1866.2
16          Italy  1860.7
17        Uruguay  1854.5
18      Australia  1843.1
19         Norway  1838.9
20        Belgium  1838.7
21    South Korea  1834.9
22           Iran  1832.7
23    Switzerland  1829.8
24        Senegal  1826.8
25        Denmark  1819.2
26  United States  1819.1
27        Algeria  1818.4
28         Canada  1817.0
29       Paraguay  1813.6
30        Austria  1792.7


In [4]:
wc2026_groups = {
    "A": ["Mexico", "South Korea", "Czech Republic", "South Africa"],
    "B": ["Canada", "Bosnia and Herzegovina", "Switzerland", "Qatar"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

print("WC 2026 teams — Elo ratings:")
print("-" * 50)
for group, teams in wc2026_groups.items():
    print(f"\n  Group {group}:")
    for team in teams:
        elo = final_ratings.get(team, 1500)
        rank = ratings_df[ratings_df["team"] == team].index
        rank_str = str(rank[0]) if len(rank) > 0 else "?"
        print(f"    {team:<30} Elo: {elo:>7.1f}  Rank: #{rank_str}")

WC 2026 teams — Elo ratings:
--------------------------------------------------

  Group A:
    Mexico                         Elo:  1888.3  Rank: #13
    South Korea                    Elo:  1834.9  Rank: #21
    Czech Republic                 Elo:  1733.5  Rank: #41
    South Africa                   Elo:  1644.0  Rank: #71

  Group B:
    Canada                         Elo:  1817.0  Rank: #28
    Bosnia and Herzegovina         Elo:  1610.4  Rank: #85
    Switzerland                    Elo:  1829.8  Rank: #23
    Qatar                          Elo:  1602.0  Rank: #88

  Group C:
    Brazil                         Elo:  1920.2  Rank: #9
    Morocco                        Elo:  1926.0  Rank: #7
    Haiti                          Elo:  1635.3  Rank: #75
    Scotland                       Elo:  1763.0  Rank: #36

  Group D:
    United States                  Elo:  1819.1  Rank: #26
    Paraguay                       Elo:  1813.6  Rank: #29
    Australia                      Elo:  1843.1 

In [5]:
elo_path     = PROCESSED_DIR / "elo_history.parquet"
ratings_path = PROCESSED_DIR / "elo_ratings_current.csv"

elo_df.to_parquet(elo_path, index=False)
ratings_df.to_csv(ratings_path, index=False)

print(f"Saved Elo history        → {elo_path}")
print(f"Saved current ratings    → {ratings_path}")
print(f"\nTop 5 sanity check:")
print(ratings_df.head(5).to_string())
print("\nNext: 04_feature_engineering.ipynb")

Saved Elo history        → ..\data\processed\elo_history.parquet
Saved current ratings    → ..\data\processed\elo_ratings_current.csv

Top 5 sanity check:
        team     elo
1      Spain  2054.0
2  Argentina  2035.7
3     France  1989.2
4    England  1950.1
5   Portugal  1934.9

Next: 04_feature_engineering.ipynb
